In [ ]:
import requests
import os
import pandas as pd
from connec_functions import GDB
import requests
import time

from accessibility import check_endpoint

### Interoperability Assessment of EuroArgo SPARQL Endpoint

### Endpoint

In [11]:
# The given endpoint is the human-facing (UI) SPARQL Endpoint
browser_endpoint = "https://co.ifremer.fr/co/argo-linked-data/html/Argo-HTML-SPARQL/"

In [12]:
# The machine-facing (API) SPARQL Endpoint is:
cli_endpoint = "https://sparql.ifremer.fr/argo/query"
gdb = GDB(cli_endpoint, "endpoint_queries")

##### Table Of Content

- [Exploring the SPARQL endpoint](#exploring-the-sparql-endpoint)
- [Technical interoperability](#technical-interoperability)
    - [SPARQL protocol support](#sparql-protocol-support)
    - [Content negotiation](#content-negotiation)
    - [Format support](#format-support) 
    - [Endpoint connectivity and performance](#endpoint-connectivity-and-performance)
- [Semantic interoperability](#semantic-interoperability)
    - [Use of standard vocabularies](#use-of-standard-vocabularies)
    - [Ontology usage and semantic alignment](#ontology-usage-and-semantic-alignment)
    - [Use of linked data principles (URIs)](#use-of-linked-data-principles-uris)
    - [Multilingual support](#multilingual-support)
- ([note on Data Granularity](#note-on-data-granularity))



### Exploring the SPARQL endpoint
*Involves getting to understand the underlying RDF graph structure*

In [13]:
# Test browser/UI endpoint vs. CLI/API endpoint
params = {
    "query": "SELECT * WHERE { ?s ?p ?o } LIMIT 1"
}
headers = {
    "Accept": "application/sparql-results+json"
}

resp_ui = requests.get(browser_endpoint, params=params, headers=headers)
print("UI endpoint status:", resp_ui.status_code)

resp_api = requests.get(cli_endpoint, params=params, headers=headers)
print("API endpoint status:", resp_api.status_code)

UI endpoint status: 403
API endpoint status: 200


In [14]:
# general exploration
gdb.execute_to_df("general.sparql")

,s,p,o
0,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,http://www.w3.org/ns/dcat#Catalog
1,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,https://co.ifremer.fr/co/argo-linked-data/doc/...
2,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://www.w3.org/2000/01/rdf-schema#label,aoml
3,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://purl.org/dc/terms/description,\n Catalog of the Argo data...
4,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://purl.org/dc/terms/publisher,http://www.argodatamgt.org/Data-Mgt-Team/ADMT-...
5,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://purl.org/dc/terms/title,aoml Argo DAC metadata
6,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://www.w3.org/ns/dcat#dataset,https://fleetmonitoring.euro-argo.eu/float/190...
7,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://www.w3.org/ns/dcat#dataset,https://fleetmonitoring.euro-argo.eu/float/190...
8,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://www.w3.org/ns/dcat#dataset,https://fleetmonitoring.euro-argo.eu/float/190...
9,https://argo.ucsd.edu/data/data-from-gdacs#aoml,http://www.w3.org/ns/dcat#dataset,https://fleetmonitoring.euro-argo.eu/float/190...


**Datamodel**

following diagram of the datamodel available through browser_endpoint:

![argo_ontology_prototype.png](https://co.ifremer.fr/co//argo-linked-data/doc/argo_ontology_prototype.png)

**Instance example**  
The underlying RDF graph structure was also examined through both the cli_endpoint and browser_endpoint by sequentially executing multiple SPARQL queries;  
this is documented in the following example instance diagram:

![euroargo datamodel diagram](./images/euroargo_datamodel-Dataset_Platform_ArgoFloat.drawio.png)

### Technical interoperability

#### SPARQL protocol support 
*Involves checking the version and supported features*

To fully comply with the SPARQL 1.1 Protocol (per [W3C spec](https://www.w3.org/TR/sparql11-protocol/)), an endpoint must support specific HTTP request types (methods) and media types for two main functionalities:

1. SPARQL Query (Read operations)

(Support for SELECT, ASK, DESCRIBE, CONSTRUCT)
| HTTP Method | Content-Type                        | Required                             |
| ----------- | ----------------------------------- | ------------------------------------ |
| **GET**     | N/A (query via URL)                 | ✔️ Yes                               |
| **POST**    | `application/x-www-form-urlencoded` | ✔️ Yes                               |
| **POST**    | `application/sparql-query`          | ⚠️ Optional, but strongly encouraged |


- Accept headers for content negotiation should also be supported:
    - application/sparql-results+json
    - application/sparql-results+xml
    - application/rdf+xml, text/turtle, etc. for CONSTRUCT/DESCRIBE


2. SPARQL Update (Write operations)

(Support for INSERT, DELETE, LOAD, etc.)
| HTTP Method | Content-Type                | Required |
| ----------- | --------------------------- | -------- |
| **POST**    | `application/sparql-update` | ✔️ Yes   |


- Only POST is allowed for SPARQL Update — GET is not valid.
- Responses are typically empty or with status 204 No Content.


3. Content Negotiation

The endpoint must respond appropriately based on the Accept header:
- For SELECT, ASK: support application/sparql-results+json, ...+xml
- For CONSTRUCT, DESCRIBE: support at least one RDF serialization:
    - application/rdf+xml, text/turtle, application/ld+json, etc.


4. Response Codes and Compliance

- Return 200 OK for valid queries.
- Return 400 Bad Request or 500 Internal Server Error for syntax/logic errors.
- May support OPTIONS for preflight (CORS-related, not required by SPARQL spec).



Summary Table:
| Function            | HTTP Method | Content-Type                        | Required                        |
| ------------------- | ----------- | ----------------------------------- | ------------------------------- |
| Query (GET)         | `GET`       | URL parameter `?query=`             | ✔️                              |
| Query (POST)        | `POST`      | `application/x-www-form-urlencoded` | ✔️                              |
| Query (POST raw)    | `POST`      | `application/sparql-query`          | ⚠️ Optional                     |
| Update              | `POST`      | `application/sparql-update`         | ✔️                              |
| Content Negotiation | `GET/POST`  | `Accept` header for result formats  | ✔️                              |
| Preflight/CORS      | `OPTIONS`   | N/A                                 | ❌ Not part of spec, but helpful |



In [15]:
def test_read_operations(endpoint, read_type="SELECT"):
    print(f"📥 Testing SPARQL 1.1 READ Operation: {read_type}\n")

    queries = {
        "SELECT": "SELECT * WHERE { ?s ?p ?o } LIMIT 1",
        "ASK": "ASK { ?s ?p ?o }",
        "CONSTRUCT": "CONSTRUCT { <http://example.org/res> ?p ?o } WHERE { <http://example.org/res> ?p ?o }",
        "DESCRIBE": "DESCRIBE <http://example.org/res>"
    }

    if read_type not in queries:
        print(f"❌ Unsupported query type: '{read_type}'. Choose from: {list(queries.keys())}")
        return

    query = queries[read_type]

    # Adjust Accept header based on type of expected result
    accept = (
        "application/sparql-results+json"
        if read_type in ["SELECT", "ASK"]
        else "text/turtle"
    )

    try:
        r_get = requests.get(endpoint, params={"query": query}, headers={"Accept": accept})
        print("GET:", r_get.status_code, r_get.headers.get("Content-Type"))
    except Exception as e:
        print("GET Error:", e)

    try:
        r_post_form = requests.post(endpoint, data={"query": query}, headers={"Accept": accept})
        print("POST form:", r_post_form.status_code, r_post_form.headers.get("Content-Type"))
    except Exception as e:
        print("POST form Error:", e)

    try:
        r_post_raw = requests.post(endpoint, data=query, headers={
            "Content-Type": "application/sparql-query",
            "Accept": accept
        })
        print("POST raw:", r_post_raw.status_code, r_post_raw.headers.get("Content-Type"))
    except Exception as e:
        print("POST raw Error:", e)

In [16]:
test_read_operations(cli_endpoint, "SELECT")

📥 Testing SPARQL 1.1 READ Operation: SELECT

GET: 200 application/sparql-results+json; charset=utf-8
POST form: 200 application/sparql-results+json; charset=utf-8


KeyboardInterrupt: 

In [17]:
test_read_operations(cli_endpoint, "ASK")

📥 Testing SPARQL 1.1 READ Operation: ASK



KeyboardInterrupt: 

In [ ]:
test_read_operations(cli_endpoint, "CONSTRUCT")

In [ ]:
test_read_operations(cli_endpoint, "DESCRIBE")

In [18]:
def test_write_operations(endpoint_url, operation_type="INSERT DATA"):
    updates = {
        "INSERT DATA": """
            INSERT DATA {
                <http://example.org/test> <http://example.org/prop> "test_insert" .
            }
        """,
        "DELETE DATA": """
            DELETE DATA {
                <http://example.org/test> <http://example.org/prop> "test_insert" .
            }
        """,
        "DELETE/INSERT": """
            DELETE {
                <http://example.org/test> <http://example.org/prop> "old_value" .
            }
            INSERT {
                <http://example.org/test> <http://example.org/prop> "new_value" .
            }
            WHERE {
                OPTIONAL { <http://example.org/test> <http://example.org/prop> "old_value" }
            }
        """,
        "CLEAR DEFAULT": """
            CLEAR DEFAULT
        """
    }

    headers = {
        "Content-Type": "application/sparql-update"
    }

    print(f"📝 Testing SPARQL 1.1 Write Operation: {operation_type}\n")

    if operation_type not in updates:
        print(f"❌ Unsupported operation type: '{operation_type}'. Choose from: {list(updates.keys())}")
        return

    update_query = updates[operation_type]

    try:
        response = requests.post(endpoint_url, data=update_query.strip(), headers=headers, timeout=10)
        print(f"Status: {response.status_code} | Content-Type: {response.headers.get('Content-Type')}")
        if response.status_code >= 400:
            print("⚠️  Error response:", response.text[:200])
    except Exception as e:
        print("❌ Request failed:", e)



In [19]:
test_write_operations(cli_endpoint, "INSERT DATA")

📝 Testing SPARQL 1.1 Write Operation: INSERT DATA

❌ Request failed: HTTPSConnectionPool(host='sparql.ifremer.fr', port=443): Max retries exceeded with url: /argo/query (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7a34e6f2bc40>, 'Connection to sparql.ifremer.fr timed out. (connect timeout=10)'))


In [ ]:
test_write_operations(cli_endpoint, "DELETE DATA")

In [ ]:
test_write_operations(cli_endpoint, "DELETE/INSERT")

In [ ]:
test_write_operations(cli_endpoint, "CLEAR DEFAULT")

#### Content negotiation

both human-faced and machine-faced SPARQL endpoint were consulted for the assessment of this part


In [20]:
def get_header_info(endpoint):
    headers = {
        "Origin": "https://example.com"  # You can change this to any origin you want to simulate
    }

    response = requests.get(endpoint, headers=headers)
    server_headers = response.headers

    return {
        # General protocol and status checks
        "General protocol and status checks" : 
            {
            "HTTP/HTTPS Protocol": endpoint.startswith("http"),
            "Status Code 200 OK": response.status_code == 200,
            },
        # Content negotiation
        "Content negotiation" : 
            {
            "MIME Type Present": "Content-Type" in server_headers,
            "MIME Type": server_headers.get("Content-Type"),
            "Content Negotiation Support": "Vary" in server_headers or "Accept" in server_headers,
            "Vary": server_headers.get("Vary"),                         # response may change depending on 'Accept-Encoding'
            "Accept-Ranges": server_headers.get("Accept-Ranges"),       # support partial content requests (e.g., 'bytes')
            },
        # Content negotiation related headers
        "Content negotiation related headers" : 
            { 
            "Content-Length": server_headers.get("Content-Length"),             # total size of the file, if specified
            "Content-Disposition": server_headers.get("Content-Disposition"),   # tells if file is inline or attachment
            "Content-Encoding": server_headers.get("Content-Encoding"),         # compression format (e.g., gzip)
            "Accept": server_headers.get("Accept"),                             # what media types the client accepts (rare in response)
            "Accept-Encoding": server_headers.get("Accept-Encoding"),           # what compression formats the client accepts (rare in response)
            },
        # Caching-related headers
        "Caching-related headers" : 
            {
            "Cache-Control": server_headers.get("Cache-Control"),       # caching policy (e.g., no-cache, max-age)
            "ETag": server_headers.get("ETag"),                         # version identifier for the file, useful for caching and validation
            },
        # CORS related headers
        "CORS-related headers": {
            "Access-Control-Allow-Origin": server_headers.get("Access-Control-Allow-Origin"),  # which origins can access the resource, expected response: access-control-allow-origin: * OR http://test.vliz.be
        }
    }

In [21]:
get_header_info(browser_endpoint)

KeyboardInterrupt: 

In [ ]:
get_header_info(cli_endpoint)

{'General protocol and status checks': {'HTTP/HTTPS Protocol': True,
  'Status Code 200 OK': False},
 'Content negotiation': {'MIME Type Present': True,
  'MIME Type': 'text/plain;charset=utf-8',
  'Content Negotiation Support': True,
  'Vary': 'Accept,Accept-Encoding,Accept-Charset,Origin,Access-Control-Request-Method,Access-Control-Request-Headers',
  'Accept-Ranges': None},
 'Content negotiation related headers': {'Content-Length': '33',
  'Content-Disposition': None,
  'Content-Encoding': None,
  'Accept': None,
  'Accept-Encoding': None},
 'Caching-related headers': {'Cache-Control': 'must-revalidate,no-cache,no-store',
  'ETag': None},
 'CORS-related headers': {'Access-Control-Allow-Origin': 'https://example.com'}}

In [ ]:
# assess formats returned by cli_endpoint
query = "SELECT * WHERE { ?s ?p ?o } LIMIT 10"
formats = {
    "SPARQL JSON": "application/sparql-results+json",
    "SPARQL XML": "application/sparql-results+xml",
    "RDF/XML": "application/rdf+xml",
    "Turtle": "text/turtle",
    "JSON-LD": "application/ld+json"
}

for name, mime in formats.items():
    headers = {"Accept": mime}
    params = {"query": query}
    
    try:
        r = requests.get(cli_endpoint, headers=headers, params=params, timeout=10)
        status = r.status_code
        ctype = r.headers.get("Content-Type", "")
        print(f"{name:15} → {status} | {ctype}")
    except Exception as e:
        print(f"{name:15} → ERROR: {e}")

SPARQL JSON     → 200 | application/sparql-results+json; charset=utf-8
SPARQL XML      → 200 | application/sparql-results+xml
RDF/XML         → 200 | application/sparql-results+xml
Turtle          → 200 | application/sparql-results+xml
JSON-LD         → 200 | application/sparql-results+xml


#### Format support 

formats returned by `browser_endpoint` include:
- text → correct
- JSON → correct
- XML → need to check 
- CSV → correct
- TSV → correct

#### Endpoint connectivity and performance 
*Includes checking endpoint availability, connectivity, performance and timeout handling*

In [ ]:
#to-do

check ook 'http monitoring tools' in google

In [23]:
# Optional: a simple test payload (SPARQL query in POST body, for example)
# This depends on the endpoint API; here we just send an empty or small query
payload = {
    "query": "SELECT * WHERE { ?s ?p ?o } LIMIT 10"  # simple SPARQL query
}
headers = {
    "Accept": "application/sparql-results+json"
}

# Configuration
timeout_seconds = 60

def test_endpoint(url, payload=None, headers=None, timeout=5):
    results = {}
    
    start_time = time.time()
    try:
        response = requests.post(url, data=payload, headers=headers, timeout=timeout)
        elapsed = time.time() - start_time

        results["reachable"] = True
        results["status_code"] = response.status_code
        results["response_time_seconds"] = round(elapsed, 3)
        results["success"] = response.ok
        results["content_preview"] = response.text[:200]  # first 200 chars
    except requests.exceptions.Timeout:
        results["reachable"] = False
        results["error"] = f"Request timed out after {timeout} seconds"
    except requests.exceptions.ConnectionError as e:
        results["reachable"] = False
        results["error"] = f"Connection error: {e}"
    except Exception as e:
        results["reachable"] = False
        results["error"] = str(e)

    return results

# Run test
result = test_endpoint(cli_endpoint, payload=payload, headers=headers, timeout=timeout_seconds)
result

{'reachable': False, 'error': 'Request timed out after 60 seconds'}

Findings

SPARQL protocol support
- support for GET(?) and POST method (POST only retrieves this is a common observation in public-facing SPARQL endpoints)

Content negotiation and format support
- human-facing SPARQL endpoint supports various file formats: Text, JSON, XML, CSV, TSV  
- machine-facing SPARQL endpoint support main RDF formats: JSONLD, Turtle and RDF/XML 

Endpoint connectivity and performance
- cli_endpoint is available, 
- cli_endpoint can't handle many requests (had to introduce function parameters and run this notebook over large period, otherwise would get timeout-errors) 
- browser_endpoint has issues with large SPARQL queries (e.g. when LIMIT not specified: SELECT ?subject ?predicate ?object WHERE { ?subject ?predicate ?object })


to include/integrate with findings on technical interoperability above:  
- simple / general / large SPARQL queries result in a 502 error

### Semantic interoperability

Provided links relevant in this context (aside from the sparql endpoint):
- [Argo vocabulary server](https://vocab.nerc.ac.uk/search_nvs/)  
- Argo ontology: [https://www.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl](https://www.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl)  
https://service.tib.eu/webvowl/#file=argo-floats.ttl#

#### Use of standard vocabularies
Using terms from standard vocabularies and common ontologies is important for interoperability because it ensures that data exchanged between different systems carries consistent and well-defined meanings. These standardized terms create a common language that minimizes ambiguity and misunderstanding, enabling diverse datasets to be accurately linked and integrated. By adopting widely accepted ontologies, data producers and consumers can align their concepts and relationships, which facilitates seamless data exchange and supports automated reasoning. This shared semantic foundation also promotes cross-domain collaboration by allowing information from different fields or organizations to be combined effectively. In contrast, self-defined ontologies often lack broad adoption and clear alignment with established standards, making data more difficult to integrate and interpret, which ultimately hampers interoperability.

Findings
- Commonly accepted vocabularies/ontologies are being used  
*(listed in browser_endpoint)*
    - prefix dct: <http://purl.org/dc/terms/>  
    prefix foaf: <http://xmlns.com/foaf/0.1/>  
    prefix geo: <https://www.w3.org/2003/01/geo/wgs84_pos#>  
    prefix nerc: <http://vocab.nerc.ac.uk/collection/>  
    prefix owl: <http://www.w3.org/2002/07/owl#>  
    prefix prov: <https://www.w3.org/TR/prov-o/>  
    prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>  
    prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#>  
    prefix sosa: <http://www.w3.org/ns/sosa/>  
    prefix ssn: <http://www.w3.org/ns/ssn/>  
    prefix xml: <http://www.w3.org/XML/1998/namespace>  
    prefix xsd: <http://www.w3.org/2001/XMLSchema#>  

- Own ontology:
    - `prefix argo: <http://www.argodatamgt.org/argo-ontology#>`   

There are many references to skos:Concepts in `http://vocab.nerc.ac.uk/collection/...` namespace 

Note on standard terms (inside-institute vs. outside-institute)  
...  
- there is linking to externally defined standard terms = good
    - leveraging use of linked data 
- usage of internally defined predicates = good, but less good (see point 2)
    - not known by external machines (e.g. dct:title vs ifremer:thisisourname)
    - solution would require community effort to develop standard data model for described entity kinds

- identifier for publisher information can be improved, for example 'http://www.argodatamgt.org/Data-Mgt-Team/ADMT-team-and-Executive-Committee'
  - could alternatively use ROR-ID for institutes, ORC-ID for people
  - currently links to just html page (no ttl or json-ld with content negotiation), could be described as linked data

#### Ontology usage and semantic alignment 
*This Includes 'ontology and vocabulary compatibility', 'cross-dataset semantic linking', 'consistency of semantics'*

Ontology usage and semantic alignment are essential for interoperability because they provide different systems with a shared, machine-readable understanding of the concepts, relationships, and rules within data. Ontologies create a *common semantic framework* by defining vocabularies - such as classes and properties - and their precise meanings, allowing datasets from various sources to “speak the same language.” They help *disambiguate* terms that might have multiple meanings, like “temperature” or “station,” by linking them to exact definitions. Ontologies also *support reasoning* by formalizing relationships, such as subclass hierarchies and property constraints, enabling systems to infer new information and maintain consistent interpretation. Semantic alignment plays a key role in *bridging heterogeneous schemas* by mapping different terms for the same concept to common ontology terms or linking them through equivalence relations like owl:sameAs or skos:exactMatch. This alignment facilitates *cross-domain integration* - for example, connecting marine, climate, and biodiversity vocabularies - allowing comprehensive analysis across disciplines. Furthermore, by aligning ontologies and vocabularies, systems can automatically link and query datasets together, greatly *reducing the need for manual mapping* and enabling smoother data integration.


Findings:
- **Ontology and vocabulary compatibility**  
https://www.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl  

--> description of owl:ontology with identifier <http://www.argodatamgt.org/argo-ontology#>  
--> prefix used in sparql endpoint: prefix argo: <http://www.argodatamgt.org/argo-ontology#>   
--> urls of some predicates in query results:  
   - <https://co.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl#type> 
   -  <https://co.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl#datamode> 
   - <https://co.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl#age>
   - <https://co.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl#stationData>  

--> urls of some classes in query results:  
   - <https://co.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl#ActivityData>

==> all these urls redirect to the description of the owl:Ontology  
ActivityData class has URI  http://www.argodatamgt.org/argo-ontology#ActivityData in the owl:Ontology

- **Cross-dataset semantic linking**  
    - no resources that use skos:exactMatch, etc. to algin with external datasets  
    - ?s ?p ?o --> ?o often skos:Concept from an external collection   

- **consistency of semantics**  
most classes are described with correct predicates (e.g. dcat:Distribution has properties dcat:downloadURL, etc.)
    - 'group' entities (e.g.  <https://fleetmonitoring.euro-argo.eu/float/1900045#group1>) are not of some class -rather a blank node with the same predicates each time 
    - entities of class dct:Software (e.g.  <https://argo.ucsd.edu/data/argo-software-tools#WJO>) don't have any properties  
    rather:  
    https://argo.ucsd.edu/data/argo-software-tools  
    --> website that describes the argo visualization and access tools & Quality control tools  
    --> information not available in RDF  
    --> note software entity <https://argo.ucsd.edu/data/argo-software-tools/#WJO> is not found/described on that website...  
    similarly:  
    dct:publisher of a dcat:Dataset is 
    http://www.argodatamgt.org/ADMT/ADMT-and-Executive-Board
    --> website that describes the ADMT and Executive Board
    --> information not available in RDF


Sidenotes:
- Some terms (classes and predicates) could be described by already existing terms in common/general/high level ontologies  
- There are a few occurences of typo: dcat:distribution when this represents a RDF class, hence should be dcat:Distribution  
- The ontology should be "generalized" ~ aligned with other domain stakeholders and be made available under a more RI neutral namespace ~ to facilitate uptake of terms by different RI's  
--> consortium/workgroup that discussed and determines these tasks and acts as an independent autorative publisher.  
- https://co.ifremer.fr/co/argo-linked-data/doc/argo-floats.ttl#Datacenter not in the described ontology

#### Use of linked data principles (URIs)

Using Linked Data principles, particularly URIs, is vital for interoperability because URIs provide global uniqueness, ensuring that when two datasets reference the same URI (e.g., http://example.org/species/Thunnus_albacares), they refer to the exact same resource. As HTTP identifiers, URIs are resolvable, allowing users and machines to retrieve additional information—such as RDF descriptions, ontologies, or documentation—thereby reducing ambiguity. Their machine-readability enables automated linking, merging, and reasoning over RDF data. This foundation supports interoperability by allowing cross-dataset linking without naming conflicts, maintaining stable references even when data changes location, and ensuring semantic clarity through consistent links to ontologies or controlled vocabularies that define the meaning of terms across systems.


Findings:
- all subjects of triples are URIs (logical)
- large % of URIs as objects of tripels 
- note: SPARQL endpoint not performant enough to analyse all objects (limited to 10000 triples); hence findings are indicative 

In [ ]:
#since SPARQL endpoint is not performant enough we limit to 10000 triples.
subj_iri = gdb.execute_to_df("LDprinciples_subject.sparql")
subj_iri

,subject,isIRI
0,https://argo.ucsd.edu/data/data-from-gdacs#aoml,true
1,https://fleetmonitoring.euro-argo.eu/float/190...,true
2,https://fleetmonitoring.euro-argo.eu/float/190...,true
3,https://fleetmonitoring.euro-argo.eu/float/190...,true
4,https://fleetmonitoring.euro-argo.eu/float/190...,true
...,...,...
9995,https://fleetmonitoring.euro-argo.eu/float/290...,true
9996,https://fleetmonitoring.euro-argo.eu/float/290...,true
9997,https://fleetmonitoring.euro-argo.eu/float/290...,true
9998,https://fleetmonitoring.euro-argo.eu/float/290...,true


In [ ]:
# % of IRIs 
subj_iri['isIRI'] = subj_iri['isIRI'].astype(str).str.lower().str.strip() == 'true'
percentage_isIRI = subj_iri['isIRI'].mean() * 100
print(f"Percentage of isIRI = True: {percentage_isIRI:.2f}%")

Percentage of isIRI = True: 100.00%


In [ ]:
obj_datatype = gdb.execute_to_df("LDprinciples_object_datatype.sparql")
obj_datatype

,object,uri,datatype
0,http://www.w3.org/ns/dcat#Catalog,true,NaN
1,https://co.ifremer.fr/co/argo-linked-data/doc/...,true,NaN
2,aoml,false,http://www.w3.org/2001/XMLSchema#string
3,\n Catalog of the Argo data...,false,http://www.w3.org/2001/XMLSchema#string
4,http://www.argodatamgt.org/Data-Mgt-Team/ADMT-...,true,NaN
...,...,...,...
9995,https://fleetmonitoring.euro-argo.eu/float/190...,true,NaN
9996,https://fleetmonitoring.euro-argo.eu/float/190...,true,NaN
9997,https://fleetmonitoring.euro-argo.eu/float/190...,true,NaN
9998,https://fleetmonitoring.euro-argo.eu/float/190...,true,NaN


In [ ]:
# % of IRIs 
obj_datatype['uri'] = obj_datatype['uri'].astype(str).str.lower().str.strip() == 'true'
percentage_isIRI = obj_datatype['uri'].mean() * 100
print(f"Percentage of isIRI = True: {percentage_isIRI:.2f}%")

# % of each datatype
unique_datatypes = obj_datatype['datatype'].unique()
for dt in unique_datatypes:
    percentage = (obj_datatype['datatype'] == dt).mean() * 100
    print(f"Percentage of datatype = {dt}: {percentage:.2f}%")


Percentage of isIRI = True: 99.63%
Percentage of datatype = nan: 0.00%
Percentage of datatype = http://www.w3.org/2001/XMLSchema#string: 0.29%
Percentage of datatype = http://www.w3.org/2001/XMLSchema#dateTime: 0.08%


#### Multilingual support 
Multilingual support ensures that data meaning is preserved and accessible across different languages and cultural contexts, which is essential for interoperability. By using RDF features such as rdfs:label and skos:prefLabel with language tags (e.g., @en, @fr, @es), terms can have human-readable equivalents in multiple languages while sharing the same URI, preserving shared meaning across systems. This approach facilitates data discovery by allowing search and query interfaces to return results in a user’s preferred language without losing semantic accuracy, and it aligns vocabularies globally by reducing the risk of mismatches when integrating data from separate national or regional repositories.  

Findings:  
- query returns empty result 
- no support for (multiple) languages 

In [ ]:
#same sparql query passed through machine facing endpoint (cli_endpoint)
gdb.execute_to_df("multilinguality.sparql")

""


**Note on Data Granularity**

In SPARQL endpoints, data granularity refers to the level of detail in the RDF triples, which influences the ability of different systems and stakeholders to integrate, interpret, and reuse the data.

**How it matters in context of interoperability**  
In ocean observations, data can range from raw sensor readings every second to monthly aggregated summaries. The SPARQL endpoint’s granularity affects interoperability in these ways:
1. Matching with external datasets  
    - Fine-grained data (e.g., per-measurement triples for salinity, temperature, timestamp, location) aligns well with other detailed datasets—making it easier to merge and cross-query.
    - Coarse-grained data (e.g., “average temperature for June at Station X”) may align only with similarly aggregated datasets, limiting integration.

2. Supporting diverse user needs
    - Scientists might need raw, high-frequency measurements for detailed modelling.
    - Policy makers may only need coarser summaries.
    - If the endpoint only offers one granularity, some users will have to post-process data, reducing interoperability in practice.

3. Preserving semantic richness  
Fine-grained RDF can encode relationships explicitly—such as this salinity reading was taken by sensor A at depth 5 m at 2025-08-01T12:00 UTC.
That explicitness improves semantic interoperability because the context travels with the data.

For ocean observation data, granularity is a balancing act. The finer the detail in the SPARQL endpoint, the easier it is to integrate with heterogeneous external datasets and support a variety of use cases—at the cost of storage and query performance. Coarser granularity improves speed and simplicity but narrows interoperability to datasets with matching aggregation levels.  

- data granularity: file level ?? to check